In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "portfolio_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/2.1_portfolio_mcp_server.py"],
        },
        "stoxly": {
            "transport": "streamable_http",
            "url": "https://www.stoxlyonline.com/api/mcp",
        },
    }
)


In [4]:
tools = await client.get_tools()

In [5]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print("---")

get_portfolio
Get the list of companies in the SRTG portfolio.
---
analyze_stock
Free fundamental analysis of a publicly traded stock. Scores the company against Stoxly's 10-point checklist (P/E, PEG, price/book, revenue growth, ROE, operating margin, return on assets, quick ratio, debt/equity, free cash flow yield) and returns the score, a descriptive verdict, every metric value and a link to the full analysis page.
---
analyze_etf
Free analysis of a US-listed ETF. Scores the fund against Stoxly's 10-point checklist (expense ratio, fund size, fund age, number of holdings, top-10 weight, top sector weight, volatility, 1/3/5-year returns) and returns the score, a descriptive verdict, every metric value and a link to the full analysis page.
---


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

agent = create_agent(
    model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a portfolio agent.No follow up questions."
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Bagaimana kondisi fundamental perusahaan-perusahaan portfolio kami?")]},
    config
    )

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Bagaimana kondisi fundamental perusahaan-perusahaan portfolio kami?', additional_kwargs={}, response_metadata={}, id='ca762e60-1e15-4472-b498-bcbc5c99e48a'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_portfolio', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'45a8a429-fc4b-4959-99f8-629299df664f': 'Cv0FARFNMg/shj3FXosv3QYpw7qhDJONeB64vaLO6audV0VT921EckNKhsG9iYkDvzb3ws9ZSR616Xky9oKt+24+h791mqL4lYMOpTIH6EPWpJxA+OxUpF2nEMfZ28m6NlTsz32gc0vQ/mpByCx7FtqLuL147QPNBCR9pniKatrfNyR2dlm6KWNYJiMs/Hh0DkWR1WPg1qw/4SOeLmCnoW8LxNfftg4AgvHAuvEyIfGGyUBKJudC1dX68H2D61t6bfUQ9EpT8p8Femwzf1BylTEBG1564ITx3Jynue5tLHBaVHd4EQzLyUaqijDtx5PrtaEnZifzHJNi//WoCeqvgTBUFBs4YSm6LTHed7Eyj1VpY8Eyr51nMF+4eLPMLtvCiF2b3lpyjJIcJ90H5qfQpCwYvjiWhaOhvsyhvKacmZZSoXmGiaTqhMgu9Stmxw5l53h4+S/jFxEtWT6VWb8hEPWBmxTGn0MTAAXHNi1UzlVKxNUHiFAZ5hX/EZePdjeVghMuNJaotkKVYXFM3Xn0nTBm1Is8h/ouhbf9b9EE4a4Uxv9Xp8sUE6CK/EgJi4DAdoz3cez9tJxSl1wdOTcMgXp

In [9]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Berikut adalah kondisi fundamental perusahaan-perusahaan dalam portofolio Anda:\n\n*   **TBIG, MDKA, ADRO, MPMX, NRCA**: Analisis fundamental tidak dapat dilakukan karena data pasar tidak tersedia.\n*   **AADI**: Memenuhi 3 dari 10 kriteria fundamental dengan penilaian "Profitabilitas Lemah". Anda dapat melihat analisis lengkapnya di: https://www.stoxlyonline.com/analysis/AADI\n*   **AGII**: Memenuhi 0 dari 10 kriteria fundamental dengan penilaian "Data Tidak Cukup". Anda dapat melihat analisis lengkapnya di: https://www.stoxlyonline.com/analysis/AGII', 'extras': {'signature': 'CtgDARFNMg95HrkC+ssqekysM3jD1oAB256/H4qwftDkvflc/OeNsM3NaJLD4MiKF/3um9WLdT+HilTDDIZWlbhYGJ/czOs3/+Z0nxLBJOFcgWwrgxSBsbVbDG3l9r41xT6ODXbKTiTBhO2B2aT4FoJLn/QJUCEklAGZItIZ0BFP5C230/AhloVALeIYHVbdXnhNLqMmLbBtblpR7zv9HJsyWxeBW1tmajtp9JP1W2nDJZNg13KsRCALsZWMrAooyQAix7ybsq1wxSqfAkgdudMAddhSUIU2VuOBO0a2yvGaI4IPchf1fvGYVivH1v62swn78G97yreyxfYOw+0WmDZkFOwo7OuT2o2Mf2GNCkte2w+VxAHlafAC1kqEwirgsgaf

In [11]:
from IPython.display import display, Markdown

answer = response["messages"][-1].content

markdown = f"""
# 📊 Saratoga Portfolio Analysis

### Fundamental Overview

{answer}

---

**Powered by:** Gemini + LangChain + MCP  
**MCP Servers:** Portfolio MCP + Stoxly MCP
"""

display(Markdown(markdown))


# 📊 Saratoga Portfolio Analysis

### Fundamental Overview

[{'type': 'text', 'text': 'Berikut adalah kondisi fundamental perusahaan-perusahaan dalam portofolio Anda:\n\n*   **TBIG, MDKA, ADRO, MPMX, NRCA**: Analisis fundamental tidak dapat dilakukan karena data pasar tidak tersedia.\n*   **AADI**: Memenuhi 3 dari 10 kriteria fundamental dengan penilaian "Profitabilitas Lemah". Anda dapat melihat analisis lengkapnya di: https://www.stoxlyonline.com/analysis/AADI\n*   **AGII**: Memenuhi 0 dari 10 kriteria fundamental dengan penilaian "Data Tidak Cukup". Anda dapat melihat analisis lengkapnya di: https://www.stoxlyonline.com/analysis/AGII', 'extras': {'signature': 'CtgDARFNMg95HrkC+ssqekysM3jD1oAB256/H4qwftDkvflc/OeNsM3NaJLD4MiKF/3um9WLdT+HilTDDIZWlbhYGJ/czOs3/+Z0nxLBJOFcgWwrgxSBsbVbDG3l9r41xT6ODXbKTiTBhO2B2aT4FoJLn/QJUCEklAGZItIZ0BFP5C230/AhloVALeIYHVbdXnhNLqMmLbBtblpR7zv9HJsyWxeBW1tmajtp9JP1W2nDJZNg13KsRCALsZWMrAooyQAix7ybsq1wxSqfAkgdudMAddhSUIU2VuOBO0a2yvGaI4IPchf1fvGYVivH1v62swn78G97yreyxfYOw+0WmDZkFOwo7OuT2o2Mf2GNCkte2w+VxAHlafAC1kqEwirgsgaf1WB9ht8t/nazqN5X/YXxwrDpRWB/Bq6g1tGAYPlaK3TFnt0uU0TEQw5IdXjDAlKVsU7Bi2G2M0NREAdMGlinedSnduoX/x7JUOlFlER97D1q1sN6HVt3hqOLEAdx7655MQ4JPrfgrEdhEt8MgDF5zOFersq6bnQC2gp/F5MjL1mVXNXmP0zJfWNZLuMODjx3ZxF/dP9oRtDfe2Au2Zp+QyYPJ2zXYQbqjNQ7mFftAidhm+mS8G91BQ=='}}]

---

**Powered by:** Gemini + LangChain + MCP  
**MCP Servers:** Portfolio MCP + Stoxly MCP


In [12]:
for message in response["messages"]:
    print(type(message).__name__)
    print(message)
    print("=" * 80)

HumanMessage
content='Bagaimana kondisi fundamental perusahaan-perusahaan portfolio kami?' additional_kwargs={} response_metadata={} id='ca762e60-1e15-4472-b498-bcbc5c99e48a'
AIMessage
content='' additional_kwargs={'function_call': {'name': 'get_portfolio', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'45a8a429-fc4b-4959-99f8-629299df664f': 'Cv0FARFNMg/shj3FXosv3QYpw7qhDJONeB64vaLO6audV0VT921EckNKhsG9iYkDvzb3ws9ZSR616Xky9oKt+24+h791mqL4lYMOpTIH6EPWpJxA+OxUpF2nEMfZ28m6NlTsz32gc0vQ/mpByCx7FtqLuL147QPNBCR9pniKatrfNyR2dlm6KWNYJiMs/Hh0DkWR1WPg1qw/4SOeLmCnoW8LxNfftg4AgvHAuvEyIfGGyUBKJudC1dX68H2D61t6bfUQ9EpT8p8Femwzf1BylTEBG1564ITx3Jynue5tLHBaVHd4EQzLyUaqijDtx5PrtaEnZifzHJNi//WoCeqvgTBUFBs4YSm6LTHed7Eyj1VpY8Eyr51nMF+4eLPMLtvCiF2b3lpyjJIcJ90H5qfQpCwYvjiWhaOhvsyhvKacmZZSoXmGiaTqhMgu9Stmxw5l53h4+S/jFxEtWT6VWb8hEPWBmxTGn0MTAAXHNi1UzlVKxNUHiFAZ5hX/EZePdjeVghMuNJaotkKVYXFM3Xn0nTBm1Is8h/ouhbf9b9EE4a4Uxv9Xp8sUE6CK/EgJi4DAdoz3cez9tJxSl1wdOTcMgXpDDx4y5ToJ01HCFeAUjAuJjjSxWsIhdlRav